In [1]:
from core import utils, database, paths
import pandas as pd
import datetime
import requests
from plyer import notification


In [2]:

today = datetime.date.today().strftime('%Y-%m-%d')


In [3]:
try:
    data : pd.DataFrame | None = database.DatabaseManager().run_query("""select * from tesco_analysts.pmajor1_wtd_for_daily_tableau_test""")
    if data is None or (hasattr(data, "empty") and data.empty):
        raise Exception("nem sikerült")
    
except Exception as e:
    import traceback
    print(f'Hiba történt: {traceback.format_exc()}')


In [4]:

data.to_csv(paths.ASSETS_PATH + f"\daily_test_data{today}.csv", index=False)


In [5]:
change_log = "This option is looks like good" 

In [6]:
payload_options={
            "format": "json",
            "temperature": 0.12,
            "top_p": 0.95,
            "num_predict": 1000,    
            "num_ctx": 4096,
            "seed": 42,
            "repeat_penalty": 1.18,
            "repeat_last_n": 256,
            "presence_penalty": 0.1,
            "frequency_penalty": 0.1,
            "stop": ['}\n\n']     
        }

In [7]:
import os
import json

def get_and_increment_version(state_file: str, key: str = "version_number", start: int = 1) -> int:
    os.makedirs(os.path.dirname(state_file), exist_ok=True)

    state = {}
    if os.path.exists(state_file):
        try:
            with open(state_file, "r", encoding="utf-8") as f:
                state = json.load(f)
        except json.JSONDecodeError:
            state = {}

    current = int(state.get(key, start - 1))
    new_value = current + 1
    state[key] = new_value

    with open(state_file, "w", encoding="utf-8") as f:
        json.dump(state, f, ensure_ascii=False, indent=2)

    return new_value

In [8]:
version_state_path = os.path.join(paths.ASSETS_PATH, "ai_daily_summary_state.json")
version_number = get_and_increment_version(version_state_path, start=21)

In [9]:
def analyse_financial_summaries(system_prompt, user_data, model='martain7r/finance-llama-8b:q4_k_m',options=payload_options):
    url = "http://localhost:11434/api/chat" # Fontos: a /api/chat végpontot használjuk!
    
    
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_data}
        ],
        "stream": False,
        "options": options
}
    response = requests.post(url, json=payload)
    if response.status_code == 200:
        
        return response.json().get("message", {}).get("content", "Nothing to analyze")
    else:
        return f"Error: {response.status_code} - {response.text}"

In [10]:
from core import utils, database, paths
import pandas as pd
import datetime
import os

# 1. ADATELŐKÉSZÍTÉS
data = pd.read_csv(paths.ASSETS_PATH + fr"\daily_test_data{today}.csv")
data['day_dt'] = pd.to_datetime(data['day'].astype(str), format='%Y%m%d')
max_day = data['day_dt'].max()

actual_df = data[data['day_dt'] == max_day]
trend_df = data[data['day_dt'] > (max_day - datetime.timedelta(days=21))]

def get_complete_metrics(df: pd.DataFrame):
    """Kiszámolja az összesített, országos és osztály szintű mutatókat."""
    departments = df['department_name'].unique()
    def calc_lfl_info(ty_col, ly_col, input_df=None):
        target_df = input_df if input_df is not None else df
        ty_sum = target_df[ty_col].sum()
        ly_sum = target_df[ly_col].sum()
        if ly_sum == 0: return "0.0% flat"
        val = (ty_sum / ly_sum - 1) * 100
        direction = "increase" if val >= 0 else "decrease"
        return f"{val:.1f}% {direction}"

    # --- TOTAL CE SZINT ---
    total_sales = df[['sales_ex_vat_ty_standard', 'sales_ex_vat_ty_promo', 'sales_ex_vat_ty_clearance']].sum().sum()
    total_margin = df[['margin_ty_standard', 'margin_ty_promo', 'margin_ty_clearance']].sum().sum()
    budget = df['sales_budget'].drop_duplicates().sum()
    forecast = df['daily_sales_forecast'].drop_duplicates().sum()
    
    metrics = {
        'sales': total_sales,
        'margin_rate': (total_margin / total_sales * 100) if total_sales != 0 else 0,
        'vs_budget_perc': ((total_sales / budget - 1) * 100) if budget != 0 else 0,
        'vs_budget': (total_sales - budget),
        'vs_forecast_perc' : ((total_sales / forecast - 1) * 100) if forecast != 0 else 0,
        'vs_forecast': (total_sales - forecast),
        'range_share' : (df['sales_ex_vat_ty_standard'].sum() / total_sales * 100) if total_sales != 0 else 0,
        'promo_share': (df['sales_ex_vat_ty_promo'].sum() / total_sales * 100) if total_sales != 0 else 0,
        'clearance_share': (df['sales_ex_vat_ty_clearance'].sum() / total_sales * 100) if total_sales != 0 else 0,
    }
    print(df.head())
    # --- ORSZÁGOS SZINT ---
    metrics['countries'] = {}
    for c in ['HU', 'CZ', 'SK']:
        c_df = df[df['country'] == c]
        if not c_df.empty:
            c_sales = c_df[['sales_ex_vat_ty_standard', 'sales_ex_vat_ty_promo', 'sales_ex_vat_ty_clearance']].sum().sum()
            metrics['countries'][c] = {
                'range_lfl_txt': calc_lfl_info('sales_lfl_standard', 'sales_lflb_standard', c_df),
                'promo_lfl_txt': calc_lfl_info('sales_lfl_promo', 'sales_lflb_promo', c_df),
                'clearance_lfl_txt': calc_lfl_info('sales_lfl_clearance', 'sales_lflb_clearance', c_df),
                'margin_rate': (c_df[['margin_ty_standard', 'margin_ty_promo', 'margin_ty_clearance']].sum().sum() / c_sales * 100) if c_sales != 0 else 0
            }
    print(df.head())        
  # --- DEPARTMENT SZINT ---
    metrics['department_name'] = {}
    for d in departments:
        d_df = df[df['department_name'] == d]
        if not d_df.empty:
            d_sales = d_df[['sales_ex_vat_ty_standard', 'sales_ex_vat_ty_promo', 'sales_ex_vat_ty_clearance']].sum().sum()
            metrics['department_name'][d] = {
                'range_lfl_txt': calc_lfl_info('sales_lfl_standard', 'sales_lflb_standard', d_df),
                'promo_lfl_txt': calc_lfl_info('sales_lfl_promo', 'sales_lflb_promo', d_df),
                'clearance_lfl_txt': calc_lfl_info('sales_lfl_clearance', 'sales_lflb_clearance', d_df),
                'margin_rate': (d_df[['margin_ty_standard', 'margin_ty_promo', 'margin_ty_clearance']].sum().sum() / c_sales * 100) if d_sales != 0 else 0
            } 


    # --- DEPARTMENT SZINT (Top/Bottom 3) ---
    dept_gb = df.groupby('department_name').apply(
        lambda x: (x['sales_lfl_standard'].sum() / x['sales_lflb_standard'].sum() - 1) * 100 if x['sales_lflb_standard'].sum() != 0 else 0
    ).sort_values(ascending=False)
    
    metrics['top_depts'] = dept_gb.head(3).to_dict()
    metrics['bottom_depts'] = dept_gb.tail(3).to_dict()

    return metrics


In [11]:
max_day

Timestamp('2026-02-08 00:00:00')

In [12]:

actual_m = get_complete_metrics(actual_df)
trend_m = get_complete_metrics(trend_df)



     fiscal_week       day country  department department_name  section  \
2931    f2025w50  20260208      HU         221      Stationery      222   
2932    f2025w50  20260208      CZ         221      Stationery      226   
2933    f2025w50  20260208      HU         421  Toys & Nursery      421   
2934    f2025w50  20260208      SK         434        Other GM       10   
2935    f2025w50  20260208      HU         434        Other GM       10   

     section_name  group      group_name  subgroup  ...  \
2931   Compulsory     10          Carton        10  ...   
2932    Schoolbag    530          Bottle       530  ...   
2933         Boys     10        Blasters        15  ...   
2934        Sport     10  Sports Equipm.        10  ...   
2935        Sport     10  Sports Equipm.       140  ...   

     margin_2ylflb_clearance  margin_2ylflb_others  units_2ylflb_standard  \
2931                 0.00000                   0.0                  562.0   
2932                 0.00000            

C:\Users\pmajor1\AppData\Local\Temp\ipykernel_31152\3558947664.py:72: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  dept_gb = df.groupby('department_name').apply(
C:\Users\pmajor1\AppData\Local\Temp\ipykernel_31152\3558947664.py:72: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  dept_gb = df.groupby('department_name').apply(


In [13]:

taken_outs = "Avoid narrative, storytelling, and speculation. "

system_message = ("""
You are a Senior Financial Controller. Return ONLY a raw JSON object with 5 keys: "Highlights", "Sales_vs_Forecast", "Sales_Drivers", "Margin_Analysis", "Department_Level".

STRICT RULES:
1. Each value MUST be a single string (not an array or object).
2. Each string MUST contain exactly 3 full sentences.
3. Every sentence MUST contain a numeric value and its unit (£, %, or pp) from the provided data and keep your eye on decimal formats.
4. NO markdown, NO code fences, NO preamble.
5. Use only: "Today" and "21-Day Trend".
6. Never use example numbers; use only the data provided in the prompt.
""")


# Csak azokat az adatokat adjuk át, amik kellenek a szövegbe, 
# és megmutatjuk a modellnek a várt stílust.
top_depts_str = ", ".join([f"{k} ({v:.1f}%)" for k, v in actual_m['top_depts'].items()])
bottom_depts_str = ", ".join([f"{k} ({v:.1f}%)" for k, v in actual_m['bottom_depts'].items()])


# Számítsunk ki 2-3 kulcsfontosságú összefüggést Pythonban, hogy ne az LLM-nek kelljen
margin_diff = actual_m['margin_rate'] - trend_m['margin_rate']
mix_improvement = trend_m['clearance_share'] - actual_m['clearance_share']

margin_bps = int((actual_m['margin_rate'] - trend_m['margin_rate']) * 100)

# KIZÁRÓLAG a tények, nulla magyarázat a modellnek, hogy mit csináljon a tagekkel
margin_improvement = actual_m['margin_rate'] - trend_m['margin_rate']
clearance_drop = trend_m['clearance_share'] - actual_m['clearance_share']
sales_gap = actual_m['sales'] - (trend_m['sales']/21)

dept_order = [
    'Gardening', 'Home', 'Stationery', 'WIGIG', 'Other GM', 'Car',
    'Home Entertainment', 'Toys & Nursery', 'DIY', 'Home seasonal',
    'Shopping bag', 'Electrical', 'Sport'
]

department_lfl_block = "\n".join([
    f"- {d}: Range {actual_m['department_name'][d]['range_lfl_txt']}, "
    f"Promo {actual_m['department_name'][d]['promo_lfl_txt']}, "
    f"Clearance {actual_m['department_name'][d]['clearance_lfl_txt']}"
    for d in dept_order
    if d in actual_m['department_name']
])


# Itt visszatesszük a teljes kontextust



user_message = f"""
### PERFORMANCE DATA (Facts – do not create new numbers):
- Net Sales: {actual_m['sales']:,.0f}£ (21-Day Trend avg: {trend_m['sales']/21:,.0f}£)
- Budget Variance: {actual_m['vs_budget']:,.2f}£ (Trend: {trend_m['vs_budget']:,.2f}£)
- Forecast Variance: {actual_m['vs_forecast']:,.2f}£ (Trend: {trend_m['vs_forecast']:,.2f}£)
- Margin Rate: {actual_m['margin_rate']:.2f}% (Trend: {trend_m['margin_rate']:.2f}%) → Improvement vs trend: {actual_m['margin_rate'] - trend_m['margin_rate']:.2f} pp ({margin_bps} bps)
- Sales Mix Today (Range/Promo/Clearance): {actual_m['range_share']:.1f}% / {actual_m['promo_share']:.1f}% / {actual_m['clearance_share']:.1f}%
- Sales Mix Trend (Range/Promo/Clearance): {trend_m['range_share']:.1f}% / {trend_m['promo_share']:.1f}% / {trend_m['clearance_share']:.1f}%
- Clearance share change (Trend → Today): {trend_m['clearance_share'] - actual_m['clearance_share']:.1f} pp
- Sales gap vs trend average: {actual_m['sales'] - (trend_m['sales']/21):,.0f}£

### REGIONAL LFLs (Text provided; do not alter numbers inside text):
- HU: Range {actual_m['countries']['HU']['range_lfl_txt']}, Promo {actual_m['countries']['HU']['promo_lfl_txt']}, Clearance {actual_m['countries']['HU']['clearance_lfl_txt']}
- CZ: Range {actual_m['countries']['CZ']['range_lfl_txt']}, Promo {actual_m['countries']['CZ']['promo_lfl_txt']}, Clearance {actual_m['countries']['CZ']['clearance_lfl_txt']}
- SK: Range {actual_m['countries']['SK']['range_lfl_txt']}, Promo {actual_m['countries']['SK']['promo_lfl_txt']}, Clearance {actual_m['countries']['SK']['clearance_lfl_txt']}

### COUNTRY MARGIN RATES (Do not infer if missing elsewhere; use exactly these):
- HU Margin Rate: {actual_m['countries']['HU']['margin_rate']:.2f}%
- CZ Margin Rate: {actual_m['countries']['CZ']['margin_rate']:.2f}%
- SK Margin Rate: {actual_m['countries']['SK']['margin_rate']:.2f}%

### DEPARTMENTS (Top/Bottom by LFL standard %):
- Top Performers: {top_depts_str}
- Bottom Performers: {bottom_depts_str}

### DEPARTMENT LFLs (Text provided; do not alter numbers inside text):
- Analyze the following LFL numbers and highlight vhich department's lfl contributed mainly and which pulled down the total LFL's.
- Write 2 sentences for each departmets about their performances, what was good and whats are the opportunities.
- {department_lfl_block}

### MANDATORY EXECUTION RULE:
Values must be NARRATIVE SENTENCES. NO bullet points, NO newlines (\n), NO dashes.

### EXAMPLE OF EXPECTED FORMAT (NOT YOUR DATA):
"Highlights": "Sales reached 100£ which is above the 80£ trend. Budget variance was 5% today. Margin dilution was avoided with a 2pp mix shift."

### LOGIC MAPPING (MANDATORY):
- Highlights: S1: Today Sales vs Trend avg. 
              S2: Budget Variance. 
              S3: Forecast Variance.
- Sales_vs_Forecast: S1: Forecast Variance. 
                     S2: Absolute gap in £ vs Trend avg. S3: Trend avg baseline in £.
- Sales_Drivers: S1: Highlight the country with best performance and answer the question where the growth from come (promo, range, clearance). 
                 S2: Discover, where is the sales miss based on the Clearane, Promo and Range LFLs.
- Margin_Analysis: S1: Margin uplift pp where the growth is coming (Promo, Clearance, Range). 
                   S2: Today's Clearance mix % vs Trend %. S3: Margin dilution term + today's margin rate %.
- Department_Level:
    S1: Highlight departments (department_name) with positive LFLs and give insight into where the growth comes from (range, promo, clearance).
    S2: For bottom performers, give insight into why they performed the worst.
    S3: Overall, identify which department was the best and which was the worst, and explain why.

MANDATORY: NO lists, NO placeholders, NO example numbers. Return exactly 3 sentences per key.
"""


In [14]:
# AI elemzés
total_summary = analyse_financial_summaries(
    system_prompt=system_message, 
    user_data=user_message,
    model='martain7r/finance-llama-8b:q4_k_m'
)


In [15]:
total_summary

' {\n  "Highlights": [\n    "Sales reached 904571£ which is above the 781718£ trend.",\n    "Budget variance was -13248057p",\n    "Forecast variance was -9395002pp"\n  ],\n  "Sales_vs_Forecast": [\n    "- Forecast Variance of -93%95002pp",\n    "- Absolute gap in £ vs Trend avg.: 122853£",\n    "- Trend avg baseline in £: 781718£"\n  ],\n  "Sales_Drivers": [\n    "The country with best performance is HU with a range lfl increase of 4.6%. The sales miss can be attributed to SK with a clearance lfl decrease of -18.7%",\n    "Gardening had an overall negative impact on total lfl\'s due to its large size."\n  ],\n  "Margin_Analysis": [\n    "+ Margin uplift pp where growth is coming (Promo): +20.8%",\n    "+ Today\'s Clearance mix % vs Trend %: -1.1pp",\n    "+ Margin dilution term + today\'s margin rate %: -0.29%"\n  ],\n  "Department_Level": [\n    "* Gardening performed poorly, Range LFL decreased by -39.4%, Promo increased by 86.6%, and Clearance increased by 33.9%. This resulted in t

In [16]:
# summary kiírása fájlba

def write_out(content):
    output_file = os.path.join(paths.ASSETS_PATH, f"ai_daily_summary_final_VERSION4_v{str(version_number)}.md")
    with open(output_file, "w", encoding="utf-8") as f:
        f.write(f"# Daily CE Performance Report - {max_day.date()}\n\n")
        f.write(content.strip()) # strip() eltávolítja a felesleges üres sorokat a végéről

write_out(total_summary)

In [17]:
#LOG HEADLINE
log_file=f"Payload options:\n {payload_options}\nsystem prompt:\n{system_message}\nPrompt:\n{user_message}"

In [18]:
def write_out(content,version_number=version_number):
    output_file = os.path.join(paths.ASSETS_PATH, f"ai_daily_summary_final_VERSION4_v{str(version_number)}_options.md")
    with open(output_file, "w", encoding="utf-8") as f:
        f.write(f"# Version of trial - {str(version_number)}\n\n{change_log}\n\n{payload_options}\n\n{system_message}\n\n{user_message}")
        f.write(content.strip()) # strip() eltávolítja a felesleges üres sorokat a végéről
    print(f"Report saved: {output_file}")

In [19]:
write_out(log_file)

Report saved: C:\Users\pmajor1\OneDrive - Tesco\Business Planning\automatization\ControlPanel_script_runner\framework_assets\ai_daily_summary_final_VERSION4_v46_options.md


In [20]:
notification.notify(message="Elfutott az AI cucc")

In [24]:
actual_m['vs_budget']

132480.5733929103

In [22]:
total_summary.replace('"','\n')

" {\n  \nHighlights\n: [\n    \nSales reached 904571£ which is above the 781718£ trend.\n,\n    \nBudget variance was -13248057p\n,\n    \nForecast variance was -9395002pp\n\n  ],\n  \nSales_vs_Forecast\n: [\n    \n- Forecast Variance of -93%95002pp\n,\n    \n- Absolute gap in £ vs Trend avg.: 122853£\n,\n    \n- Trend avg baseline in £: 781718£\n\n  ],\n  \nSales_Drivers\n: [\n    \nThe country with best performance is HU with a range lfl increase of 4.6%. The sales miss can be attributed to SK with a clearance lfl decrease of -18.7%\n,\n    \nGardening had an overall negative impact on total lfl's due to its large size.\n\n  ],\n  \nMargin_Analysis\n: [\n    \n+ Margin uplift pp where growth is coming (Promo): +20.8%\n,\n    \n+ Today's Clearance mix % vs Trend %: -1.1pp\n,\n    \n+ Margin dilution term + today's margin rate %: -0.29%\n\n  ],\n  \nDepartment_Level\n: [\n    \n* Gardening performed poorly, Range LFL decreased by -39.4%, Promo increased by 86.6%, and Clearance increase

In [23]:
print(total_summary)

 {
  "Highlights": [
    "Sales reached 904571£ which is above the 781718£ trend.",
    "Budget variance was -13248057p",
    "Forecast variance was -9395002pp"
  ],
  "Sales_vs_Forecast": [
    "- Forecast Variance of -93%95002pp",
    "- Absolute gap in £ vs Trend avg.: 122853£",
    "- Trend avg baseline in £: 781718£"
  ],
  "Sales_Drivers": [
    "The country with best performance is HU with a range lfl increase of 4.6%. The sales miss can be attributed to SK with a clearance lfl decrease of -18.7%",
    "Gardening had an overall negative impact on total lfl's due to its large size."
  ],
  "Margin_Analysis": [
    "+ Margin uplift pp where growth is coming (Promo): +20.8%",
    "+ Today's Clearance mix % vs Trend %: -1.1pp",
    "+ Margin dilution term + today's margin rate %: -0.29%"
  ],
  "Department_Level": [
    "* Gardening performed poorly, Range LFL decreased by -39.4%, Promo increased by 86.6%, and Clearance increased by 33.9%. This resulted in the largest share of the d